In [1]:
import os
import json
import glob
import zipfile
import urllib.request

from PIL import Image
from tqdm import tqdm
from controlnet_aux import LineartDetector, PidiNetDetector

c:\Users\Stoland\miniconda3\envs\sdxl-scribble-controlnet\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\Stoland\miniconda3\envs\sdxl-scribble-controlnet\lib\site-packages\timm\models\registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
c:\Users\Stoland\miniconda3\envs\sdxl-scribble-controlnet\lib\site-packages\controlnet_aux\segment_anything\modeling\tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_5m_224 in registry with controlnet_aux.segment_anything.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  re

In [2]:
urls = {
    "train2017.zip": "http://images.cocodataset.org/zips/train2017.zip",
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}
for name, url in urls.items():
    dst = f"data/coco/{name}"
    if not os.path.exists(dst):
        print("downloading", name); urllib.request.urlretrieve(url, dst)
    with zipfile.ZipFile(dst) as z:
        z.extractall("data/coco")

In [2]:
def square_resize(img, res):
    w, h = img.size
    s = res / min(w, h)
    img = img.resize((round(w*s), round(h*s)), Image.BICUBIC)
    w, h = img.size
    left, top = (w-res)//2, (h-res)//2
    return img.crop((left, top, left+res, top+res))

In [3]:
DETECT_RES = 512 
IMAGE_RES = 1024

os.makedirs(f"data/proc/{IMAGE_RES}/image", exist_ok=True)
os.makedirs(f"data/proc/{IMAGE_RES}/lineart", exist_ok=True)
os.makedirs(f"data/proc/{IMAGE_RES}/scribble", exist_ok=True)

In [4]:
lineart = LineartDetector.from_pretrained("lllyasviel/Annotators").to("cuda")
pidi    = PidiNetDetector.from_pretrained("lllyasviel/Annotators").to("cuda")

if hasattr(lineart, "eval"):
    lineart.eval()

if hasattr(pidi, "eval"):
    pidi.eval()

In [5]:
for path in tqdm(glob.glob("data/coco/train2017/*.jpg")):
    name = os.path.splitext(os.path.basename(path))[0]
    
    if os.path.exists(f"data/proc/{IMAGE_RES}/image/{name}.png") \
        and os.path.exists(f"data/proc/{IMAGE_RES}/lineart/{name}.png") \
        and os.path.exists(f"data/proc/{IMAGE_RES}/scribble/{name}.png"):
        continue
    
    img = square_resize(Image.open(path).convert("RGB"), IMAGE_RES)
    img.save(f"data/proc/{IMAGE_RES}/image/{name}.png")
    
    lineart(img, detect_resolution=DETECT_RES, image_resolution=IMAGE_RES).save(f"data/proc/{IMAGE_RES}/lineart/{name}.png")
    pidi(img, detect_resolution=DETECT_RES, image_resolution=IMAGE_RES, scribble=True).save(f"data/proc/{IMAGE_RES}/scribble/{name}.png")

100%|██████████| 118287/118287 [3:56:57<00:00,  8.32it/s]  
